# Clasificación de 3 escenas con ViT preentrenado en ImageNet

Segundo Transformer del reto: el mismo tipo de modelo que `ScenesViT.ipynb`
(un **ViT**, atención global sobre parches), pero partiendo de los pesos de
**ImageNet** en lugar de entrenarlo desde cero. Se usa
`vit_base_patch16_224_imagenet` de `keras-hub` (ViT-B/16: 12 bloques,
dimensión 768, 12 cabezas, ~86 M de parámetros; parches de 16×16 sobre una
entrada de 224×224 → 196 parches + el token de clase).

Con los dos ViT el análisis separa los dos factores:

* **ViT desde cero vs ViT preentrenado**: misma familia de arquitectura, solo
  cambia el preentrenamiento → cuánto vale ImageNet para un Transformer.
* **ViT preentrenado vs DenseNet121**: los dos parten de ImageNet y se ajustan
  igual → atención frente a convolución a igualdad de preentrenamiento.

El ajuste repite el esquema de `ScenesDenseNet.ipynb`:

1. **Fase 1** — backbone congelado: solo se entrena la cabeza (`Dropout` +
   `Dense(3)`) sobre el token de clase con `Adam(1e-3)`.
2. **Fase 2** — fine-tuning: se descongelan los **2 últimos bloques** del
   encoder y la `LayerNormalization` final, y se reentrena con `Adam(1e-5)`.
   El ViT no tiene `BatchNormalization`, así que no hay estadísticas que
   proteger.

**Preprocesado.** Las imágenes del split común (128×128, `uint8` sin
normalizar) se redimensionan a 224×224 y se escalan a [-1, 1]
(`norm_mean = norm_std = 0.5`, la configuración del preset) **dentro del
modelo**. Se hace con capas de Keras porque el `ViTImageConverter` de
`keras-hub` exige `tensorflow-text`, que no existe para Python 3.14.

**Pesos.** La primera ejecución descarga ~345 MB desde Kaggle. Los pesos
ajustados (`model.weights.h5`, ~345 MB) superan el límite de 100 MB por
archivo de GitHub y están **ignorados por git**: el repositorio guarda las
métricas, las predicciones y la historia, y en un clon nuevo el cuaderno
reentrena el modelo. Cambia `FORCE_RETRAIN = True` para reentrenarlo.

Requiere `keras-hub` (`pip install --no-deps keras-hub kagglehub kagglesdk
regex tqdm`; `--no-deps` evita que pip baje la versión de `keras` del
entorno).

## Configuración

In [1]:
import keras
import keras_hub
import numpy as np
import pandas as pd
import seaborn as sns
from keras import layers
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam

import common_3scenes as cs

FORCE_RETRAIN = False
PRESET = "vit_base_patch16_224_imagenet"
VIT_INPUT = 224
FINE_TUNE_BLOCKS = 2
sns.set_theme(style="whitegrid", context="notebook")
keras.utils.set_random_seed(cs.SEED)
np.random.seed(cs.SEED)

I0000 00:00:1795050124.965541  124358 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795050125.044130  124358 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1795050126.080150  124358 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Carga del dataset y split común

In [2]:
data = cs.load_split_data()
X_train = data["X_train"].astype("float32")
X_val = data["X_val"].astype("float32")
X_test = data["X_test"].astype("float32")
y_train_labels = data["y_train"]
y_val_labels = data["y_val"]
y_test_labels = data["y_test"]
test_paths = data["test_paths"]

y_train = cs.one_hot(y_train_labels)
y_val = cs.one_hot(y_val_labels)

print("Train:", X_train.shape, "| Validación:", X_val.shape, "| Test:", X_test.shape)
print("Distribución de test:", {name: int(np.sum(y_test_labels == name)) for name in cs.CLASSES})

Train: (610, 128, 128, 3) | Validación: (108, 128, 128, 3) | Test: (240, 128, 128, 3)
Distribución de test: {'coast': 90, 'forest': 82, 'highway': 68}


## Modelo, entrenamiento y evaluación

`Input(128, 128, 3)` → augmentación suave (`RandomFlip("horizontal")`,
`RandomRotation(0.05)`; solo activa en entrenamiento) → `Resizing(224, 224)` →
`Rescaling(1/127.5, offset=-1)` → backbone ViT-B/16 de ImageNet → token de
clase (posición 0 de la secuencia de 197 tokens) → `Dropout(0.2)` →
`Dense(3, softmax)`.

Fase 1: máximo 8 épocas; fase 2: máximo 6 épocas. Las dos usan batch de 32 y
`EarlyStopping` sobre `val_accuracy` (con `restore_best_weights`), igual que la
DenseNet121. En CPU cada época cuesta unos minutos (cada imagen pasa por los 12
bloques del ViT-B), por eso los topes de épocas son bajos.

In [3]:
def build_vit_pretrained():
    inputs = layers.Input(shape=(cs.IMG_SIZE, cs.IMG_SIZE, 3))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.Resizing(VIT_INPUT, VIT_INPUT, interpolation="bilinear")(x)
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(x)  # [0, 255] -> [-1, 1], as the preset

    backbone = keras_hub.models.Backbone.from_preset(PRESET)
    backbone.trainable = False

    tokens = backbone(x, training=False)
    class_token = tokens[:, 0]
    x = layers.Dropout(0.2)(class_token)
    outputs = layers.Dense(len(cs.CLASSES), activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="vit_pretrained_3scenes")
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=1e-3),
        metrics=["accuracy"],
    )
    return model


def unfreeze_last_blocks(model, n_blocks=FINE_TUNE_BLOCKS):
    backbone = next(layer for layer in model.layers if isinstance(layer, keras_hub.models.Backbone))
    backbone.trainable = True
    backbone.get_layer("vit_patching_and_embedding").trainable = False
    encoder = backbone.get_layer("vit_encoder")
    for block in encoder.encoder_layers[:-n_blocks]:
        block.trainable = False


def fit_vit_pretrained(model):
    phase1_callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ]
    history_phase1, seconds_phase1 = cs.train_keras_model(
        model, X_train, y_train, X_val, y_val,
        epochs=8, batch_size=32, callbacks=phase1_callbacks,
    )

    unfreeze_last_blocks(model)
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=1e-5),
        metrics=["accuracy"],
    )
    phase2_callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    ]
    history_phase2, seconds_phase2 = cs.train_keras_model(
        model, X_train, y_train, X_val, y_val,
        epochs=6, batch_size=32, callbacks=phase2_callbacks,
    )

    history = pd.concat(
        [pd.DataFrame(history_phase1.history), pd.DataFrame(history_phase2.history)],
        ignore_index=True,
    )
    history.index = np.arange(1, len(history) + 1)
    return model, history, seconds_phase1 + seconds_phase2


def evaluate_vit_pretrained(model, history, training_seconds):
    predicted_idx, probabilities, inference_ms = cs.predict_keras(model, X_test)
    predicted = np.array(cs.CLASSES)[predicted_idx]
    metrics = cs.compute_metrics(
        name="ViT preentrenado (ImageNet)",
        family="IA generativa (Transformer)",
        y_true=y_test_labels,
        y_pred=predicted,
        y_prob=probabilities,
        params=model.count_params(),
        trainable_params=sum(np.prod(weight.shape) for weight in model.trainable_weights),
        training_seconds=training_seconds,
        inference_ms=inference_ms,
        epochs=len(history),
        hyperparameters={
            "base": PRESET,
            "input": VIT_INPUT,
            "phase1_lr": 1e-3,
            "phase2_lr": 1e-5,
            "batch_size": 32,
            "epochs_phase1": 8,
            "epochs_phase2": 6,
            "dropout": 0.2,
            "fine_tuning_blocks": FINE_TUNE_BLOCKS,
            "trained_from_scratch": False,
        },
        notes=(
            "ViT-B/16 pretrained on ImageNet (keras-hub) with two-phase fine-tuning: "
            "frozen backbone, then the last encoder blocks unfrozen."
        ),
    )
    # The full model (~345 MB) is not saved as model.keras: only the weights,
    # which are git-ignored; load_vit_pretrained rebuilds from the preset.
    cs.save_results(
        "vit_preentrenado", metrics, y_test_labels, predicted, probabilities,
        test_paths=test_paths, history=history,
    )
    model.save_weights(cs.result_dir("vit_preentrenado") / cs.ARTIFACTS["weights"])
    return metrics


def load_vit_pretrained(folder):
    weights_path = cs.result_dir(folder) / cs.ARTIFACTS["weights"]
    if not weights_path.exists():
        return None
    model = build_vit_pretrained()
    unfreeze_last_blocks(model)  # same trainable layout as when the weights were saved
    model.load_weights(weights_path)
    return model


model, history, metrics, retrained = cs.load_or_train(
    "vit_preentrenado",
    build_model=build_vit_pretrained,
    fit_model=fit_vit_pretrained,
    evaluate_model=evaluate_vit_pretrained,
    load_model=load_vit_pretrained,
    force=FORCE_RETRAIN,
)
print("Accuracy en test:", round(metrics["accuracy"], 4))
print("Parámetros:", f"{metrics['parameters']:,}")

[train] Training model in resultados/vit_preentrenado


E0000 00:00:1795050128.100561  124358 cuda_platform.cc:57] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:50



Epoch 1/8


20/20 - 127s - 6s/step - accuracy: 0.7672 - loss: 0.5908 - val_accuracy: 0.9352 - val_loss: 0.1664 - learning_rate: 0.0010


Epoch 2/8


20/20 - 114s - 6s/step - accuracy: 0.9869 - loss: 0.0895 - val_accuracy: 0.9537 - val_loss: 0.0870 - learning_rate: 0.0010


... (21 outputs left out)

Epoch 4: early stopping


Restoring model weights from the end of the best epoch: 1.


Results saved to: resultados/vit_preentrenado


Accuracy en test: 0.975
Parámetros: 85,800,963


## Observaciones

* Es la misma familia de modelo que `ScenesViT.ipynb`; la diferencia de
  accuracy entre los dos ViT mide lo que aporta el preentrenamiento de
  ImageNet a un Transformer con ~610 imágenes de train.
* Frente a DenseNet121 (también ImageNet + fine-tuning en dos fases) la
  comparación es justa: queda solo la diferencia de arquitectura, atención
  global frente a convolución con conectividad densa.
* Los artefactos quedan en `resultados/vit_preentrenado/` (`metrics.json`,
  `predictions.csv`, `history.csv`, `classification_report.txt`; los pesos,
  `model.weights.h5`, solo en local). La comparación con el resto de
  modelos, en `reto-1.ipynb`.